# Fundamentals of Data Science - Network Science Lab

# Lab Session 02 - Network Visualization

## The `igraph` Library

`igraph` is a high-performance library for large-scale network analysis and visualization.

For the main `Graph` class, typical high-level methods include:
- `vcount()`, which returns the number of vertices of the graph.
- `ecount()`, which returns the number of vertices of the graph.
- `degree()`, which provides vertex degrees.
- `components()`, which returns the connected components within the graph.
- `layout()`, which allows to set the coordinate layout for visualization.

In [ ]:
!pip install -q igraph powerlaw matplotlib numpy

In [ ]:
import os
import igraph as ig
import powerlaw
import matplotlib.pyplot as plt
import numpy as np
import random
import pandas as pd

In [ ]:
# Create a toy graph and check its structural properties.
g = ig.Graph(edges=[(0,1), (1,2), (2,3), (3,4)], directed=False)

# Print a summary of the graph.
print(g.summary())

# Analyse the structure of the graph.
print("Number of vertices:", g.vcount())
print("Number of edges:", g.ecount())
print("Degrees:", g.degree())

# Compute connected components
components = g.components()
print("Number of components:", len(components))
print("Component sizes:", [len(c) for c in components])

In [ ]:
# Assign some attributes to the nodes of the graph.
g.vs["name"] = ["Alice", "Bob", "Charlie", "David", "Eve"]

# The summary will now show attributes as well.
print("Graph Summary:", g.summary())

# Print the name of each vertex.
print("Vertices:", g.vs["name"])

# Extract the edges through an edge list or an adjacency matrix.
print("Edges:", g.get_edgelist())
g.get_adjacency()

# Lastly, plot the graph.
fig, ax = plt.subplots(figsize=(5, 4))

ig.plot(
    g,
    target=ax,
    vertex_label=g.vs["name"],
    vertex_size=30,
    vertex_color="lightblue",
    edge_width=2,
    vertex_label_dist=1.5
    )

Alternatively, one can create a graph starting from an edge list or an adjacency matrix.

In [ ]:
# Using an edge list.
edges = [(0, 1), (1, 2), (2, 3), (3, 0), (2, 4)]
g_from_edges = ig.Graph.TupleList(edges, directed=False)
print("Graph from Edgelist Edges:", g_from_edges.get_edgelist())

fig, ax = plt.subplots(figsize=(5, 4))

ig.plot(
    g_from_edges,
    target=ax,
    vertex_label=g.vs["name"],
    vertex_size=30,
    vertex_color="lightblue",
    edge_width=2,
    vertex_label_dist=1.5
    )

In [ ]:
# Using an adjacency matrix.
matrix = [
    [0, 1, 1, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 0],
    [0, 1, 0, 0]
]
g_from_matrix = ig.Graph.Adjacency(matrix, mode="undirected")
g_from_matrix.vs["name"] = list(range(g_from_matrix.vcount()))

print("Graph from Adjacency Matrix - Summary:", g_from_matrix.summary())

fig, ax = plt.subplots(figsize=(5, 4))

ig.plot(
    g_from_matrix,
    target=ax,
    vertex_label=g.vs["name"],
    vertex_size=30,
    vertex_color="lightblue",
    edge_width=2,
    vertex_label_dist=1.5
    )

Furthermore, the `igraph` library allows to create graphs starting from well-known structures, such as the Erdős-Rényi or the Barabási-Albert networks.

In [ ]:
# Set a random seed for reproducibility.
random.seed(2411)

# Create the two graphs.
g_er = ig.Graph.Erdos_Renyi(n=100, m=200, directed=False) # 100 vertices and 200 edges.
g_ba = ig.Graph.Barabasi(n=100, m=2, directed=False) # 100 vertices and 2 edges for each new node.

print(g_er.summary())
print(g_ba.summary())

# Plot the two graphs side-by-side.
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

for ax, g, title in zip(axes, [g_er, g_ba], ["Erdős–Rényi", "Barabási–Albert"]):
  ig.plot(g,
          target=ax,
          vertex_color="lightblue",
          vertex_size=10,
          edge_width=1.5)
  ax.set_title(title)

plt.tight_layout()
plt.show()

## Importing Graphs

In [ ]:
from google.colab import files

### Using a TXT File

In [ ]:
# Lines starting with % are comments and must be skipped by Pandas..
# Note that sep=r"\s+" means "split on any amount of whitespace".
edges_frozen = pd.read_csv("frozen_interactions.txt", sep=r"\s+", comment="%",
                    header=None, names=["from", "to", "weight"], dtype=str)
print(edges_frozen.head())

In [ ]:
# Build a graph using the `TupleList` method, which builds a graph from (source, target) pairs and attributes.
g_frozen = ig.Graph.TupleList(edges_frozen.itertuples(index=False),
                       directed=False, edge_attrs=["weight"])

print(g_frozen.summary())
print("\n")
print(g_frozen.vs["name"])
print("\n")
print(g_frozen.es["weight"])
print("\n")
print(g_frozen.is_weighted())

In [ ]:
# Rank the nodes in the graph by their strength.
g_frozen.es["weight"] = [float(w) for w in g_frozen.es["weight"]] # Remember that weight will be a string.

g_frozen.vs["strength"] = g_frozen.strength(weights="weight")
top = sorted(g_frozen.vs, key=lambda v: v["strength"], reverse=True)[:10]

# Create a subgraph containing just the top 10 nodes and the edges between them.
sub = g_frozen.induced_subgraph([v.index for v in top])

# Plot the graph using the Fruchterman-Reingold layout, which draws nodes closer to stronger edges.
max_w = max(sub.es["weight"])
fig, ax = plt.subplots(figsize=(8, 8))
ig.plot(sub,
        target=ax,
        layout=sub.layout_fruchterman_reingold(weights="weight"),
        vertex_label=sub.vs["name"],
        vertex_size=[8 + 0.1 * s for s in sub.vs["strength"]],
        vertex_color="lightblue",
        edge_width=[0.5 + 4 * w / max_w for w in sub.es["weight"]],
        vertex_label_dist = 1)
ax.set_title("Top 10 Nodes by Strength")
plt.show()

In [ ]:
# The same ranking can also be stored as a table for readability.
print([(v["name"], v["strength"]) for v in top])

### Using a CSV File.

The Zebra network is an undirected, unweighted network that contains interactions between 28 Grévy's zebras in Kenya.

In this graph, each node represents a zebra and an edge between two zebras shows that there was an interaction between them during the study.

In [ ]:
# Importing the graph as an edge list.
el = pd.read_csv("zebra_edgelist.csv", dtype=str)
g_el = ig.Graph.TupleList(el.itertuples(index=False), directed=False)

# Importing the graph as an adjacency matrix.
A = pd.read_csv("zebra_adjmatrix.csv", index_col=0)
g_am = ig.Graph.Adjacency(A.values.tolist(), mode="undirected")
g_am.vs["name"] = [str(x) for x in A.index]

# Importing the graph as an adjacency list.
al = pd.read_csv("zebra_adjlist.csv", dtype=str)
al_edges = [(row.node, nb)
            for row in al.itertuples()
            for nb in str(row.neighbors).split(";") if nb != "nan"]
g_al = ig.Graph.TupleList(al_edges, directed=False)
g_al.simplify() # Since each edge is listed twice (once per endpoint), it is necessary to remove duplicates first.

# Take a look at each summary of the network.
print("As an edge list.")
print("Edge list rows :", len(el))
print(el.head())

print("As an adjacency matrix.")
print("Adjacency matrix size:", A.shape, "=", A.size, "cells")
print(A.head())

print("As an adjacency list")
print("Adjacency list rows:", len(al))
print(al.head())

In [ ]:
# Alternatively, one can just use .DataFrame() when building a graph.

# Setting `use_vids=True`, which is the default choice, interprets values as vertex IDs.
# Setting `use_vids=False` interprets values as vertex names.
g_zebra = ig.Graph.DataFrame(el, directed=False, use_vids=False)

# Take a look at the summary of the network.
print(g_zebra.summary())
print(g_zebra.vs["name"][:5])

The Bison network is a directed, weighted network that indicates dominance display between American bison in 1972 on the National Bison Range in Moiese (Montana).

In this graph, each node represents a bison and an edge represents dominance of a bison over the other.

In particular, the edge values indicate how often this dominance behaviour was observed.

In [ ]:
# Importing the graph as an edge list.
el_bison = pd.read_csv("bison_edgelist.csv")
el_bison.head()

# Importing the graph as an adjacency matrix.
am_bison = pd.read_csv("bison_adjmatrix.csv", index_col=0)
am_bison.head()

# Importing the graph as an adjacency list.
g_bison_el = ig.Graph.DataFrame(el_bison, directed=True, use_vids=False)
g_bison_el.summary()

In [ ]:
# When working with an adjacency matrix, it is necessary to use `ig.Graph.Weighted_Adjacency()` for weights.
g_bison_am = ig.Graph.Weighted_Adjacency(am_bison.values.tolist(), mode="directed")
g_bison_am.summary()

# Alternatively, it is possible to represent the adjacency matrix using numpy, although the labels will be discarded.
np_am = np.genfromtxt("bison_adjmatrix.csv", delimiter=",", skip_header=1)[:, 1:]
print(np_am.shape, np.isnan(np_am).any())   # should be (n, n) and False
print(np_am[0:2])
g_bison_np = ig.Graph.Weighted_Adjacency(np_am.tolist(), mode="directed")
g_bison_np.summary()

### Using the GraphML Format

GraphML is an XML-based file format for graphs that can store all information about a graph in a single file.

For instance, the TrumpWorld Dataset provides a comprehensive map of the business connections, investments, and corporate relationships surrounding Donald J. Trump.


In [ ]:
g_trump = ig.Graph.Read_GraphML("trumpworld.graphml")
g_trump.summary()

# Analyse the summary of the network.
print(g_trump.summary())
print("Nodes:", g_trump.vcount())
print("Edges:", g_trump.ecount())
print("Directed:", g_trump.is_directed())
print("Weighted:", g_trump.is_weighted())
print("Density:", round(g_trump.density(), 5))
print("Connected:", g_trump.is_connected())

# Take a look at the attributes stored in the file.
print("Node attributes:", g_trump.vs.attributes())
print("Edge attributes:", g_trump.es.attributes())

# For instance, take a look at the first node and the first edge.
print(g_trump.vs[0]["id"])
print(g_trump.vs[0].attributes())
print(g_trump.es[0].attributes())

In [ ]:
# Find the most connected entities within the graph.
deg = pd.Series(g_trump.degree(), index=g_trump.vs["id"], name="degree")
print(deg.sort_values(ascending=False).head(10))

In [ ]:
# Search for vertices containing a certain name in their ID (case-insensitive)

to_search = "giuliani"
searched_vertices = [v for v in g_trump.vs if to_search in v["id"].lower()]

if not searched_vertices:
  print("No vertex containing '"+ to_search +"' was found.")
else:
  for v in searched_vertices:
    print(f"Found vertex: ID = {v['id']}, Index = {v.index}, Attributes = {v.attributes()}")

# Take the matching vertex and get its neighborhood.
target_v = searched_vertices[1]
neighbors = g_trump.neighbors(target_v.index)
subgraph_nodes = list(set([target_v.index] + neighbors))

# Construct the neighborhood subgraph.
sub = g_trump.induced_subgraph(subgraph_nodes)
print(f"\nSubgraph constructed with {sub.vcount()} nodes and {sub.ecount()} edges.")

# Plot the neighborhood subgraph.
fig, ax = plt.subplots(figsize=(10, 10))
ig.plot(
    sub,
    target=ax,
    vertex_label=sub.vs["id"],
    vertex_size=15,
    vertex_color=["red" if val == target_v["id"] else "lightblue" for val in sub.vs["id"]],
    edge_width=1.5,
    vertex_label_size=10,
    vertex_label_dist=1.5
)
ax.set_title(f"Neighborhood Subgraph for {target_v['id']}")
plt.show()

## Saving Graphs

In [ ]:
# Extract the edge list and the adjacency matrix associated to the graph.
edgelist_trump = g_trump.get_edgelist()
adj_trump = g_trump.get_adjacency()

labels = [str(x) for x in g_trump.vs["id"]]

# Using numpy, the labels will be discarded.
M_trump = np.array(adj_trump.data, dtype=np.int32)
E_trump = np.array(edgelist_trump, dtype=np.int32)

# Using Pandas, the labels are kept.
df_trump = pd.DataFrame(M_trump, index=labels, columns=labels)
df_edges = pd.DataFrame(
    [(labels[s], labels[t]) for s, t in edgelist_trump],
    columns=["source", "target"]
)

# Save the graph as a CSV file.
df_trump.to_csv("trump_adjmatrix.csv")

# Save the graph as a TXT file.
np.savetxt("trump_adjmatrix.txt", M_trump, fmt="%d", delimiter=" ")
with open("trump_labels.txt", "w") as f:
    f.write("\n".join(labels))

# Save the graph as an edge list.
df_edges.to_csv("trump_edgelist.csv", index=False) # As a CSV file.
np.savetxt("trump_edgelist.txt", E_trump, fmt="%d", delimiter=" ") # As a TXT file.

# Save the graph as a parquet file, which improves efficiency through binarization and compression.
df_trump.rename_axis("node").reset_index().to_parquet("trump_adjmatrix.parquet")

# Save the graph as a GraphML file.
g_trump.write_graphml("trump_copy.graphml")

# Compare file sizes.
for f in ["trump_adjmatrix.csv", "trump_adjmatrix.txt", "trump_labels.txt",
          "trump_adjmatrix.parquet",
          "trump_edgelist.csv",
          "trump_edgelist.txt",
          "trump_copy.graphml"]:
    print(f"{f:28s} {os.path.getsize(f) / 1e6:8.2f} MB")

## Power Laws

A power law is a statistical distribution that follows the form $p(x) \propto x^{-\alpha}$, for some fixed $\alpha$.

In particular,
- If $\alpha < 2$, the distribution has infinite mean and variance.
- If $2 \leq \alpha \leq 3$, which is the most common case when dealing with real-life networks, the distribution has finite mean but infinite variance.
- If $\alpha > 3$, the distribution has finite mean and finite variance.

In [ ]:
# Consider again the TrumpWorld network.
deg = pd.Series(g_trump.degree(), index=g_trump.vs["id"], name="degree")
print(deg.sort_values(ascending=False).head(50))

# Analyse and plot the frequency of the degrees within the graph.
degrees = np.sort(np.array(g_trump.degree()))

print(degrees)

fig, ax = plt.subplots(figsize=(8, 8))

ax.hist(degrees,
        bins=range(0, degrees.max()+1),
        color = "red",
        edgecolor = "black")

ax.set_title("Frequency of Degrees (Histogram)")
ax.set_xlabel("Degree")
ax.set_ylabel("Frequency")

plt.show()

In [ ]:
# Apply a log-log scale to better visualize the power law pattern.
fig, ax = plt.subplots(figsize=(5, 4))
vals, counts = np.unique(degrees, return_counts=True)

ax.loglog(vals, counts, "o", color="red")
ax.set_title("Log-Log Visualization")
ax.set_xlabel("Degree"); ax.set_ylabel("Frequency")

plt.show()

In [ ]:
# Visualize the distribution using its empirical cumulative distribution function.
degrees = np.array(g_trump.degree())
x = np.sort(degrees)
n = len(x)

# Find the unique degree values and check how many nodes have each of these degrees.
vals, counts = np.unique(degrees, return_counts=True)

# Take the running total of counts, which denotes the number of nodes with degree at most each value, and divide by n to normalize counts into a probability.
ecdf = counts.cumsum() / n

# Plot the empirical cumulative distribution function of the distribution.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(vals, ecdf)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("ECDF of Degree")
ax.set_xlabel("Degree")
ax.set_ylabel("P(X ≤ x)")
plt.show()

In [ ]:
# However, when dealing with heavy-tailed data distributions, the empirical complementary cumulative distribution function typically provides more information.
degrees = np.array(g_trump.degree())
x = np.sort(degrees)
n = len(x)

# Check the first occurrence of each unique degree, which provides the exact P(X >= x).
vals, first_idx = np.unique(x, return_index=True)
eccdf = 1 - first_idx / n

# Plot the empirical complementary cumulative distribution function of the distribution.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(vals, eccdf)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("ECCDF of Degree")
ax.set_xlabel("Degree")
ax.set_ylabel("P(X ≥ x)")
plt.show()

In [ ]:
degrees = np.array(g_trump.degree())

plt.figure(figsize=(6, 4))
fig = powerlaw.plot_ccdf(degrees, color="black")
fig.set_ylabel("P(X >= x)")
fig.set_xlabel("Frequency")
plt.show()

## Exercise

Using the edge list representation, analyse one network from the Facebook100 Dataset.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load the edge list.
edgelist = pd.read_csv(
    "Stanford_edgelist.csv",
    names=["source", "target"]
)

print(edgelist.head())
print(edgelist[["source", "target"]].min().min())

edgelist["source"] = edgelist["source"] - 1
edgelist["target"] = edgelist["target"] - 1

print(edgelist.head())

In [ ]:
# Load the metadata.
metadata = pd.read_csv("Stanford_metadata.csv")
print(metadata.head())

metadata = metadata.reset_index()
metadata = metadata.rename(columns={"index": "name"})
print(metadata.head())

In [ ]:
# Plot the network.
g = ig.Graph.DataFrame(edgelist, directed=False, vertices = metadata, use_vids = True)

In [ ]:
# Study the summary and structure of the network.
print(g.summary())
print(f"Nodes: {g.vcount()}")
print(f"Edges: {g.ecount()}")

In [ ]:
# Visualize the degree distribution through the empirical complementary cumulative distribution function.
degrees = np.array(g.degree())
x = np.sort(degrees)
n = len(x)

# Check the first occurrence of each unique degree, which provides the exact P(X >= x).
vals, first_idx = np.unique(x, return_index=True)
eccdf = 1 - first_idx / n

# Plot the empirical complementary cumulative distribution function of the distribution.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(vals, eccdf)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("ECCDF of Degree")
ax.set_xlabel("Degree")
ax.set_ylabel("P(X ≥ x)")
plt.show()